# PetCare AI — Colab 노트북 (RAG → LangGraph)

Cornell 수의학 문서 기반 **RAG** 를 먼저 완성하고, 그 위에 **LangGraph** 상담 파이프라인을
얹어 검증하는 노트북입니다. 명세 44절의 22개 섹션을 순서대로 담았습니다.

## 사용법

1. **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요.
   기본 임베딩 모델(`BAAI/bge-m3`)은 CPU 에서 매우 느립니다(실측: 283문서 / 2,460 chunk → **CPU 626초**).
2. 셀을 **위에서 아래로 순서대로** 실행합니다. 뒤 셀은 앞 셀의 변수에 의존합니다.
3. 준비물
   - `raw.zip` (Cornell 문서 JSON) — 3번 섹션에서 업로드
   - `petcare_ai/` 패키지 — 4번 섹션에서 업로드 / Drive 마운트 / git clone 중 택 1
   - `OPENAI_API_KEY` (선택) — 없어도 **전 셀이 규칙 기반으로 끝까지 동작**합니다
   - `TAVILY_API_KEY` (선택) — 없으면 웹 fallback 은 mock 으로만 시연합니다
   - `LANGSMITH_API_KEY` (선택) — trace 확인용

## 이 노트북이 지키는 원칙

| 원칙 | 근거 |
| --- | --- |
| API 키를 코드에 하드코딩하지 않는다 | 명세 7·47절 |
| LLM/Tavily 키가 없어도 규칙 기반으로 전 셀이 실행된다 | 구현 가이드 0-1절 |
| dog/cat FAISS index 를 분리한다 | 명세 11절 |
| Tavily 는 RAG 가 부족할 때만 호출한다 | 명세 15·47절 |
| 웹 근거는 allowlist 검증을 통과한 것만 쓴다 | 명세 15절 |
| 기존 index 는 설정 지문이 같으면 재사용한다 | 명세 44절 |
| 확정 진단·처방·실시간 진료 가능 단정을 생성하지 않는다 | 명세 40·47절 |

## LLM 설정 (확정)

- provider = `openai`, model = **`gpt-5.4-mini`** (기존 일기장·진단서 노트북과 동일 모델)
- 환경 변수 = `OPENAI_API_KEY`

---
## 1. 패키지 설치

버전을 고정하지 않습니다(명세 7절). Colab 기본 패키지와 충돌하면 import 경로가 깨지기 때문에,
현재 설치되는 호환 버전을 쓰고 deprecated import 는 사용하지 않습니다.

- `langchain` 메타 패키지와 `langchain-community` 는 넣지 않습니다.
  `vector_store` 는 faiss + numpy 만으로 직접 구현되어 있어 필요가 없습니다.
- provider 는 **OpenAI 하나만** 설치합니다(`langchain-openai`).

In [ ]:
# 1) 패키지 설치 — Colab 런타임에서 최초 1회만 실행하면 됩니다.
import subprocess
import sys

IN_COLAB = False
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    pass

PACKAGES = [
    "langgraph",
    "langsmith",
    "langchain-core",
    "langchain-text-splitters",
    "langchain-openai",       # provider = openai (gpt-5.4-mini)
    "langchain-huggingface",  # 임베딩 래퍼
    "sentence-transformers",  # bge-m3 / e5 백엔드
    "faiss-cpu",
    "pydantic",
    "pandas",
    "numpy",
    "pytest",
    "tavily-python",
    "reportlab",
]

if IN_COLAB:
    print("Colab 환경 — 패키지를 설치합니다. (수 분 소요)")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *PACKAGES],
        check=False,
    )
    print("설치 완료. 설치 후 'RESTART SESSION' 안내가 뜨면 런타임을 다시 시작하고 이 셀부터 이어서 실행하세요.")
else:
    print("Colab 이 아니므로 자동 설치를 건너뜁니다. 필요하면 아래를 직접 실행하세요:")
    print("  pip install " + " ".join(PACKAGES))

In [ ]:
# 2) 설치 상태 점검 — 없는 패키지는 그 기능만 건너뛰고 나머지는 계속 실행됩니다.
import importlib

CHECK = {
    "pydantic": "필수 (스키마)",
    "numpy": "필수 (벡터)",
    "faiss": "필수 (FAISS index)",
    "langgraph": "필수 (LangGraph)",
    "pandas": "표 출력 (없으면 텍스트로 대체)",
    "sentence_transformers": "임베딩 (없으면 deterministic 백엔드로 대체)",
    "langchain_huggingface": "임베딩 래퍼",
    "langchain_openai": "LLM (없으면 규칙 기반)",
    "tavily": "웹 fallback (없으면 mock 만)",
    "reportlab": "PDF 생성",
    "langsmith": "trace",
}

for name, note in CHECK.items():
    try:
        importlib.import_module(name)
        print(f"  [OK]   {name:<24} {note}")
    except Exception as exc:
        print(f"  [없음] {name:<24} {note}  ({type(exc).__name__})")

---
## 2. Secret · API Key 설정

**키를 코드에 하드코딩하지 않습니다**(명세 7·47절). 읽는 순서는

1. 이미 설정된 환경 변수
2. Colab 좌측 🔑 **보안 비밀(Secrets)** — `userdata.get(...)`
3. `getpass` 입력 (Enter 로 건너뛸 수 있음)

**키가 하나도 없어도 이 노트북은 끝까지 실행됩니다.**
`build_llm()` 이 `None` 을 돌려주고 모든 Agent 가 규칙 기반 경로로 동작하도록 설계돼 있습니다.

In [ ]:
import getpass
import os


def read_secret(name: str, *, prompt: bool = False, description: str = "") -> bool:
    """환경변수 → Colab Secret → getpass 순으로 키를 읽어 os.environ 에 넣는다.

    반환값은 '키가 준비됐는가'. 값 자체는 절대 출력하지 않는다.
    """
    value = os.environ.get(name)

    if not value:
        try:
            from google.colab import userdata  # type: ignore[import-not-found]

            value = userdata.get(name)
        except Exception:
            value = None

    if not value and prompt:
        try:
            entered = getpass.getpass(f"{name} 입력 ({description} / Enter 로 건너뛰기): ")
            value = entered.strip() or None
        except Exception:
            value = None

    if value:
        os.environ[name] = str(value).strip()
        return True
    os.environ.pop(name, None)
    return False


# 명세 7절 환경 변수 — Anthropic 은 쓰지 않습니다(provider=openai 확정).
HAS_OPENAI = read_secret("OPENAI_API_KEY", prompt=True, description="LLM. 없으면 규칙 기반")
HAS_TAVILY = read_secret("TAVILY_API_KEY", prompt=True, description="웹 fallback. 없으면 mock")
HAS_LANGSMITH = read_secret("LANGSMITH_API_KEY", prompt=True, description="trace. 없어도 무방")

LLM_PROVIDER = "openai"      # 명세 7절: provider 는 설정으로 교체 가능
LLM_MODEL = "gpt-5.4-mini"   # 기존 일기장·진단서 노트북과 동일 모델 — 임의로 바꾸지 마세요

print(f"OPENAI_API_KEY   : {'있음' if HAS_OPENAI else '없음 → LLM 없이 규칙 기반으로 실행'}")
print(f"TAVILY_API_KEY   : {'있음' if HAS_TAVILY else '없음 → 웹 fallback 은 mock 으로만 시연'}")
print(f"LANGSMITH_API_KEY: {'있음' if HAS_LANGSMITH else '없음 → trace 비활성'}")
print(f"LLM              : provider={LLM_PROVIDER} / model={LLM_MODEL}")

---
## 3. raw.zip 업로드 및 압축 해제

`raw.zip` 안에 `raw/cornell_pet_health_documents.json` 이 들어 있습니다.
이미 파일이 있으면 다시 업로드하지 않습니다.

> 압축을 풀지 않고 zip 에서 바로 읽는 경로도 있습니다:
> `service.build_index(zip_path="raw.zip")`

In [ ]:
import zipfile
from pathlib import Path

WORK_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
RAW_ZIP = WORK_DIR / "raw.zip"
DATA_DIR = WORK_DIR / "raw"
DOCUMENTS_FILENAME = "cornell_pet_health_documents.json"
DOCUMENTS_PATH = DATA_DIR / DOCUMENTS_FILENAME

print("작업 디렉터리:", WORK_DIR)

# (1) 이미 풀려 있으면 아무것도 하지 않는다.
if DOCUMENTS_PATH.exists():
    print(f"이미 존재합니다: {DOCUMENTS_PATH} ({DOCUMENTS_PATH.stat().st_size:,} bytes)")
else:
    # (2) zip 이 없으면 업로드받는다.
    if not RAW_ZIP.exists():
        try:
            from google.colab import files  # type: ignore[import-not-found]

            print("raw.zip 을 업로드하세요.")
            uploaded = files.upload()
            for name in uploaded:
                src = WORK_DIR / name
                if src.exists() and src != RAW_ZIP and name.lower().endswith(".zip"):
                    src.replace(RAW_ZIP)
        except Exception as exc:
            print(f"업로드 위젯을 쓸 수 없습니다({exc}). raw.zip 을 {WORK_DIR} 에 직접 두세요.")

    # (3) 압축 해제
    if RAW_ZIP.exists():
        with zipfile.ZipFile(RAW_ZIP) as zf:
            members = zf.namelist()
            print(f"raw.zip 항목 {len(members)}개 (앞 5개): {members[:5]}")
            zf.extractall(WORK_DIR)
        print("압축 해제 완료.")
    else:
        print("raw.zip 을 찾지 못했습니다.")

# (4) 최종 확인 — 다른 위치에 풀렸을 수도 있으니 한 번 더 탐색한다.
if not DOCUMENTS_PATH.exists():
    found = list(WORK_DIR.rglob(DOCUMENTS_FILENAME))
    if found:
        DOCUMENTS_PATH = found[0]
        DATA_DIR = DOCUMENTS_PATH.parent
        print(f"다른 경로에서 찾았습니다: {DOCUMENTS_PATH}")

if DOCUMENTS_PATH.exists():
    print(f"문서 JSON 준비 완료: {DOCUMENTS_PATH} ({DOCUMENTS_PATH.stat().st_size:,} bytes)")
else:
    print("[경고] 문서 JSON 이 없습니다. 5번 섹션부터는 실행할 수 없습니다.")

---
## 4. 프로젝트 모듈 연결 (`petcare_ai` 패키지)

명세 8절은 `%%writefile` 로 모듈을 노트북 안에 인라인하는 것을 허용하지만,
**이 프로젝트의 `petcare_ai` 는 40개 파일 / 약 2만 줄** 이라 노트북에 인라인하면
셀 하나가 수천 줄이 되어 읽을 수도 유지할 수도 없습니다.
그래서 **저장소의 패키지를 그대로 가져와 쓰는 방식**을 씁니다 — 노트북과 서버가
같은 코드를 공유하므로 드리프트도 생기지 않습니다.

아래 **방법 A / B 중 하나만** 실행한 뒤, 마지막 검증 셀로 `import petcare_ai` 를 확인하세요.

### 방법 A (권장) — 폴더 업로드 또는 Google Drive 마운트

In [ ]:
# 방법 A-1: petcare_ai 폴더를 통째로 업로드
#   Colab 좌측 파일 탭 → 폴더 아이콘 → petcare_ai 폴더 드래그 앤 드롭
#   (또는 petcare_ai.zip 으로 압축해 업로드한 뒤 아래에서 풀기)
import zipfile
from pathlib import Path

PKG_ZIP = WORK_DIR / "petcare_ai.zip"
if PKG_ZIP.exists() and not (WORK_DIR / "petcare_ai").exists():
    with zipfile.ZipFile(PKG_ZIP) as zf:
        zf.extractall(WORK_DIR)
    print("petcare_ai.zip 압축 해제 완료.")

# 방법 A-2: Google Drive 마운트 (Drive 에 petcare_ai 를 올려 둔 경우)
MOUNT_DRIVE = False          # ← True 로 바꾸면 Drive 를 마운트합니다.
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/PetCare-AI/jewon-ai"

if MOUNT_DRIVE:
    try:
        from google.colab import drive  # type: ignore[import-not-found]

        drive.mount("/content/drive")
        print("Drive 마운트 완료. 프로젝트 경로:", DRIVE_PROJECT_DIR)
    except Exception as exc:
        print(f"Drive 마운트 실패: {exc}")

### 방법 B — git clone

In [ ]:
# 방법 B: 저장소를 clone 해서 jewon-ai 디렉터리를 프로젝트 루트로 씁니다.
import subprocess
import sys
from pathlib import Path

USE_GIT_CLONE = False        # ← True 로 바꾸면 clone 을 시도합니다.
REPO_URL = ""                # 예: "https://github.com/<org>/PetCare-AI.git"
CLONE_DIR = WORK_DIR / "PetCare-AI"

if USE_GIT_CLONE:
    if not REPO_URL:
        print("REPO_URL 을 먼저 채우세요.")
    elif CLONE_DIR.exists():
        print(f"이미 clone 되어 있습니다: {CLONE_DIR}")
    else:
        result = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)],
            capture_output=True,
            text=True,
        )
        print(result.stdout or result.stderr)
else:
    print("USE_GIT_CLONE=False — 방법 A(업로드/Drive)를 사용합니다.")

### 검증 — 어느 방법을 썼든 `import petcare_ai` 가 되는지 확인

In [ ]:
# petcare_ai 가 있는 디렉터리를 찾아 sys.path 에 넣고 import 를 검증합니다.
import importlib
import sys
from pathlib import Path

CANDIDATE_ROOTS = [
    WORK_DIR,
    WORK_DIR / "PetCare-AI" / "jewon-ai",
    Path("/content/drive/MyDrive/PetCare-AI/jewon-ai"),
    Path.cwd(),
    Path.cwd().parent,
]

PROJECT_ROOT = None
for candidate in CANDIDATE_ROOTS:
    try:
        if (candidate / "petcare_ai" / "__init__.py").exists():
            PROJECT_ROOT = candidate.resolve()
            break
    except OSError:
        continue

if PROJECT_ROOT is None:
    raise RuntimeError(
        "petcare_ai 패키지를 찾지 못했습니다.\n"
        "  · 방법 A: petcare_ai 폴더(또는 petcare_ai.zip)를 업로드했는지 확인\n"
        "  · 방법 B: USE_GIT_CLONE=True 와 REPO_URL 설정 후 위 셀 재실행\n"
        f"  · 탐색한 경로: {[str(p) for p in CANDIDATE_ROOTS]}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import os

os.chdir(PROJECT_ROOT)          # pytest·상대경로 출력 기준을 프로젝트 루트로 맞춘다.

import petcare_ai

importlib.reload(petcare_ai)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("petcare_ai   :", petcare_ai.__file__)
print("version      :", petcare_ai.__version__)

# 하위 모듈이 실제로 import 되는지도 확인한다(무거운 의존성은 지연 import 라 여기서 안 끌려온다).
for module_name in (
    "petcare_ai.config",
    "petcare_ai.schemas",
    "petcare_ai.llm",
    "petcare_ai.rag.loader",
    "petcare_ai.rag.normalizer",
    "petcare_ai.rag.chunker",
    "petcare_ai.rag.service",
    "petcare_ai.adapters.clinical_data_adapter",
    "petcare_ai.graph.state",
):
    importlib.import_module(module_name)
print("하위 모듈 import 확인 완료.")

In [ ]:
# 전역 설정 — 경로·LLM·임베딩을 한 곳에서 확정합니다(config.Settings).
from petcare_ai.config import get_settings
from petcare_ai.llm import build_llm, llm_available

settings = get_settings()
settings.data_dir = DATA_DIR
settings.documents_filename = DOCUMENTS_PATH.name
settings.index_dir = PROJECT_ROOT / "faiss_index"
settings.output_dir = PROJECT_ROOT / "outputs"
settings.llm_provider = LLM_PROVIDER
settings.openai_model = LLM_MODEL
settings.environment = "colab"
settings.ensure_dirs()

# LLM — 키가 없으면 None 이 돌아오고, 모든 Agent 가 규칙 기반으로 동작합니다.
llm = build_llm(settings)

print("documents_path :", settings.documents_path)
print("index_dir      :", settings.index_dir)
print("output_dir     :", settings.output_dir)
print("llm_provider   :", settings.llm_provider, "/", settings.llm_model)
print("llm_available  :", llm_available(settings), "→ build_llm() =", type(llm).__name__ if llm else "None (규칙 기반)")

---
## 5. RAG 데이터 검증 (명세 9절)

`load_documents()` 는 **비정상 문서를 삭제하지 않고 `errors` 에 기록**합니다.
중복 제거 우선순위는 `content_hash` → `id` → `source_url` 이고,
중복을 합칠 때 **`categories` 는 합집합으로 병합**합니다(같은 문서가 서로 다른 카테고리로
두 번 등재된 경우가 실제로 4쌍 있습니다 — 버리면 정보 손실).

실측 기대값: 원본 **287건**(dog 160 / cat 127) → 중복 제거 후 **283건**(dog 160 / cat 123).

In [ ]:
from petcare_ai.rag.loader import load_documents

documents, load_report = load_documents(settings.documents_path)

print(load_report.summary())
print()
print(f"유효 문서 수(중복 제거 후): {load_report.total_valid}")
print(f"오류/경고 기록: {len(load_report.errors)}건")
for line in load_report.errors[:10]:
    print("  -", line)
if len(load_report.errors) > 10:
    print(f"  ... 외 {len(load_report.errors) - 10}건")

In [ ]:
# 데이터 통계 표 (명세 18절 산출물 1)
from collections import Counter


def show_table(rows, title=""):
    """pandas 가 있으면 DataFrame, 없으면 텍스트로 출력한다."""
    if title:
        print(f"\n=== {title} ===")
    if not rows:
        print("(결과 없음)")
        return None
    try:
        import pandas as pd

        frame = pd.DataFrame(rows)
        try:
            from IPython.display import display

            display(frame)
        except Exception:
            print(frame.to_string(index=False))
        return frame
    except Exception:
        headers = list(rows[0].keys())
        print(" | ".join(headers))
        for row in rows:
            print(" | ".join(str(row.get(h, "")) for h in headers))
        return None


species_counter = Counter(str(doc.get("species")) for doc in documents)
category_counter = Counter(
    category for doc in documents for category in (doc.get("categories") or [])
)

show_table(
    [
        {"항목": "원본 문서 수", "값": load_report.total_raw},
        {"항목": "유효 문서 수", "값": load_report.total_valid},
        {"항목": "dog", "값": load_report.dog_count},
        {"항목": "cat", "값": load_report.cat_count},
        {"항목": "제외", "값": load_report.excluded_count},
        {"항목": "중복 제거", "값": load_report.duplicate_count},
        {"항목": "평균 본문 길이(자)", "값": round(load_report.average_content_length, 1)},
        {"항목": "category 종류", "값": len(category_counter)},
    ],
    "데이터 통계",
)

show_table(
    [{"category": name, "문서 수": count} for name, count in category_counter.most_common(10)],
    "상위 category 10종",
)

emergency_categories = [
    name for name in category_counter
    if any(word in name.lower() for word in ("emergency", "toxin", "poison", "hazard", "advisor"))
]
print("\n응급 관련 category:", emergency_categories)
print("species 분포:", dict(species_counter))

---
## 6. 정규화 + chunking (명세 10절)

- 본문은 `content_markdown` 우선, 없으면 `content_text` 를 씁니다.
- heading(h2 이상) 경계로 자르고, 너무 짧은 조각은 합칩니다.
- **모든 chunk 는 출처 metadata**(`title` / `source` / `source_url` / `species` / `categories` /
  `heading_path`)를 갖습니다. 하나라도 빠지면 답변에서 출처를 제시할 수 없습니다.

In [ ]:
from petcare_ai.rag.chunker import chunk_documents, chunk_stats
from petcare_ai.rag.normalizer import normalize_documents

normalized = normalize_documents(documents)
chunks = chunk_documents(normalized, settings.rag)
stats = chunk_stats(chunks)

show_table([{"항목": k, "값": v} for k, v in stats.items()], "chunk 통계")

# 출처 metadata 누락 검사 — 하나라도 비면 RAG 답변의 출처가 깨진다.
REQUIRED_META = ("title", "source", "source_url", "species", "document_id")
missing = [
    {"chunk_id": chunk.chunk_id, "빠진 필드": key}
    for chunk in chunks
    for key in REQUIRED_META
    if not chunk.metadata.get(key)
]
print(f"\n출처 metadata 누락 chunk: {len(missing)}건", "→ 정상" if not missing else "→ 확인 필요")
if missing:
    show_table(missing[:10], "누락 상세(최대 10건)")

In [ ]:
# 샘플 chunk 1건 — metadata 전체를 눈으로 확인합니다.
import json

sample = next((c for c in chunks if c.metadata.get("heading_path")), chunks[0])
print("chunk_id   :", sample.chunk_id)
print("document_id:", sample.document_id)
print("길이       :", sample.length)
print("metadata   :")
print(json.dumps(sample.metadata, ensure_ascii=False, indent=2)[:1200])
print("\n본문(앞 400자):")
print(sample.text[:400])

---
## 7. 임베딩 및 FAISS index 생성 (명세 11·44절)

- 기본 모델 **`BAAI/bge-m3`** (dim 1024). 한국어 질문 → 영어 문서 검색 품질이 가장 좋았습니다.
- **GPU 를 권장합니다.** 실측: 283문서 / 2,460 chunk 인덱싱이 **CPU 626초**.
  GPU 가 없으면 아래 코드의 주석 한 줄로 더 가벼운 모델(`intfloat/multilingual-e5-base`)로 바꾸세요.
- **dog / cat index 를 분리**합니다(`faiss_dog/`, `faiss_cat/`). 교차 오염이 구조적으로 불가능합니다.
- **기존 index 가 있고 설정 지문(임베딩 백엔드·모델·정규화·chunk 파라미터·차원)이 같으면 재생성하지 않습니다.**
  `build_index()` 반환의 `reused` 로 확인합니다.

In [ ]:
# 임베딩 백엔드 선택
import os

# ── 모델 교체 ───────────────────────────────────────────────────────────────
# GPU 가 없어 bge-m3 인덱싱이 너무 느리면 아래 주석을 해제해 더 가벼운 모델을 쓰세요.
# settings.rag.embedding_model = "intfloat/multilingual-e5-base"
# ───────────────────────────────────────────────────────────────────────────

# sentence-transformers 를 쓸 수 없는 환경(오프라인 스모크 테스트)용 탈출구.
# True 로 두면 모델 다운로드 없이 384차원 결정론적 임베딩으로 전 파이프라인이 돕니다.
USE_DETERMINISTIC_EMBEDDINGS = False

if USE_DETERMINISTIC_EMBEDDINGS:
    settings.rag.embedding_backend = "deterministic"
    print("[주의] deterministic 임베딩 — 검색 품질 검증용이 아니라 배선 확인용입니다.")
else:
    settings.rag.embedding_backend = "huggingface"

# GPU 감지
try:
    from petcare_ai.rag.embeddings import detect_device

    device = detect_device(settings.rag.embedding_device)
except Exception as exc:
    device = "unknown"
    print("device 감지 실패:", exc)

print("embedding_backend :", settings.rag.embedding_backend)
print("embedding_model   :", settings.rag.embedding_model)
print("device            :", device)
if device != "cuda" and not USE_DETERMINISTIC_EMBEDDINGS:
    print(
        "\n[경고] GPU 가 없습니다. bge-m3 로 283문서/2,460 chunk 인덱싱은 CPU 에서 약 626초 걸립니다.\n"
        "       런타임 → 런타임 유형 변경 → T4 GPU 를 권장합니다.\n"
        "       또는 위의 embedding_model 교체 주석을 해제하세요."
    )

In [ ]:
# index 생성 또는 재사용 — 명세 44절 "설정이 달라졌을 때만 재생성"
import time

from petcare_ai.rag.service import VeterinaryRagService

FORCE_REBUILD = False   # True 로 두면 지문이 같아도 무조건 다시 만듭니다.

service = VeterinaryRagService(settings=settings, llm=llm)

started = time.time()
index_report = service.build_index(
    documents_path=settings.documents_path,
    force_rebuild=FORCE_REBUILD,
)
elapsed = time.time() - started

print(index_report.summary())
print()
show_table(
    [
        {"항목": "reused (재사용 여부)", "값": index_report.reused},
        {"항목": "reason", "값": index_report.reason},
        {"항목": "문서 수", "값": index_report.document_count},
        {"항목": "chunk 수", "값": index_report.chunk_count},
        {"항목": "index_counts", "값": str(index_report.index_counts)},
        {"항목": "소요 시간(초)", "값": round(elapsed, 1)},
    ],
    "FAISS index 생성 결과",
)

print("\nindex 저장 경로 (명세 18절 산출물 3):")
for species in ("dog", "cat"):
    path = settings.index_path(species)
    print(f"  {species}: {path}  존재={path.exists()}")

print("\nvector store 상태:", service.store.stats())

---
## 8. RAG 검색 데모 (명세 17절 예시 query 5개)

한국어 질문으로 **영어 문서**를 검색합니다.
`query_builder` 가 한↔영 증상 사전으로 영어 query 를 함께 만들고,
`retriever` 가 ko/en 두 query 결과를 chunk_id 로 중복 제거해 상위 N 건을 남깁니다.

> **ko+en 동시 검색은 필수 장치입니다.** 실측상 한국어 query 단독으로는
> "구토·무기력", "심장질환 기침" 질문에서 관련 문서가 **0/5** 였습니다.

In [ ]:
# 명세 17절 예시 query 5개 + species fixture
EXAMPLE_QUERIES = [
    ("강아지가 계속 구토하고 축 처져 있어요", "dog"),
    ("고양이가 소변을 보려고 하는데 나오지 않아요", "cat"),
    ("강아지가 숨을 힘들게 쉬고 있어요", "dog"),
    ("고양이가 백합을 먹은 것 같아요", "cat"),
    ("심장질환이 있는 강아지가 기침을 계속해요", "dog"),
]


def profile_for(species: str) -> dict:
    """검색 데모용 최소 PET 프로필 — 종이 index 선택을 결정한다."""
    if species == "cat":
        return {"id": 2, "name": "나비", "species": "cat", "breed": "korean shorthair", "age_years": 3}
    return {"id": 1, "name": "초코", "species": "dog", "breed": "poodle", "age_years": 8}


retrieval_results = {}
rows = []

for question, species in EXAMPLE_QUERIES:
    try:
        result = service.retrieve(question, profile_for(species))
    except Exception as exc:
        rows.append({"질문": question, "종": species, "오류": f"{type(exc).__name__}: {exc}"})
        continue

    retrieval_results[question] = result
    top1 = result.documents[0] if result.documents else None
    rows.append(
        {
            "질문": question,
            "종": result.species,
            "문서수": len(result.documents),
            "충분성": result.sufficiency,
            "web_fallback": result.web_fallback_required,
            "top1 제목": (top1.title[:40] if top1 else "-"),
            "top1 score": (round(top1.score, 4) if top1 and top1.score is not None else "-"),
        }
    )

show_table(rows, "query별 검색 결과 (명세 18절 산출물 4)")

In [ ]:
# 한 질문의 상세 — 출처 URL / heading metadata 가 보존되는지 확인 (명세 17절 검색 테스트)
detail_question = EXAMPLE_QUERIES[0][0]
detail = retrieval_results.get(detail_question)

if detail is None:
    print("검색 결과가 없어 상세를 표시할 수 없습니다.")
else:
    print("질문:", detail_question)
    print("빌드된 query:", service.last_debug.get("query"))
    print()
    show_table(
        [
            {
                "순위": i + 1,
                "제목": doc.title[:38],
                "score": round(doc.score, 4) if doc.score is not None else None,
                "species": doc.species,
                "heading_path": " > ".join(doc.heading_path)[:40],
                "source_url": doc.source_url[:60],
            }
            for i, doc in enumerate(detail.documents)
        ],
        f"'{detail_question}' 검색 상세",
    )

# 종 분리 검증 (명세 17절 / 43절)
cat_question = "고양이가 소변을 보려고 하는데 나오지 않아요"
cat_result = retrieval_results.get(cat_question)
if cat_result:
    species_set = {doc.species for doc in cat_result.documents}
    print(f"\n고양이 질문 결과의 species 집합: {species_set}")
    print("강아지 문서 혼입 여부:", "없음 (정상)" if species_set <= {"cat"} else "혼입! (버그)")

# 파이프라인 단계별 요약
show_table(service.debug_rows(), "RAG 파이프라인 단계별 결과")

---
## 9. 충분성 평가 + score 임계값 calibration (명세 14절)

### 왜 calibration 이 필수인가

절대 코사인 임계값은 **임베딩 모델을 바꾸면 곧바로 무의미**해집니다. 실측 근거:

| 모델 | 코퍼스 score 분포 | 관련 문서 top1 margin |
| --- | --- | --- |
| `multilingual-e5-small` | min 0.713 / mean 0.772 / max 0.808 / **std 0.013** | 밴드가 뭉개져 판별 불가 |
| **`BAAI/bge-m3`** | min 0.292 / **mean 0.473** / max 0.654 / **std 0.045** | **+0.094 ~ +0.170** |

`min_relevance_score=0.30` 같은 절대값은 bge-m3 에서 사실상 전부 통과시킵니다.
그래서 **코퍼스 평균 대비 상대 margin** 을 기본 판단 기준으로 씁니다.

### 확정값

- 관측된 top1 margin **최소값 0.094** 의 절반 수준인 **`min_relevance_margin = 0.05`** 를 보수적 기준으로 채택.
- 아래 셀에서 `calibrate_threshold()` 로 실제 분포를 뽑아 config 에 되쓰고
  **`score_threshold_calibrated=True`** 로 표시합니다.

> 참고: 백합(lily) 질문은 top1 이 오답이고 margin 도 최저(0.094)였습니다.
> **이런 질문이 `insufficient` 로 걸러져 Tavily fallback 으로 가야 정상**이며, 회귀 테스트로 보존합니다.

In [ ]:
# score 분포 calibration — 명세 14절
from petcare_ai.rag.sufficiency import apply_calibration, calibrate_threshold

documents_by_query = {q: r.documents for q, r in retrieval_results.items()}
calibration = calibrate_threshold(documents_by_query)

show_table(
    [
        {"항목": key, "값": calibration.get(key)}
        for key in (
            "query_count", "count", "min", "max", "mean", "median", "stdev",
            "p10", "p25", "p50", "p75", "p90", "top1_mean",
            "suggested_min_relevance_score", "suggested_min_relevance_margin",
        )
    ],
    "score 분포 (calibration)",
)
print("\n안내:", calibration.get("note"))

show_table(
    [
        {
            "질문": q[:34],
            "문서수": v.get("document_count"),
            "min": round(v["min"], 4) if v.get("min") is not None else None,
            "max": round(v["max"], 4) if v.get("max") is not None else None,
            "mean": round(v["mean"], 4) if v.get("mean") is not None else None,
        }
        for q, v in (calibration.get("per_query") or {}).items()
    ],
    "query별 score 요약",
)

In [ ]:
# calibration 결과를 config 에 되쓴다 → score_threshold_calibrated=True
RECOMMENDED_MARGIN = 0.05   # CALIBRATION-FINDINGS: 관측 top1 margin 최소 0.094 의 절반 (보수적)

apply_calibration(calibration, settings)

measured_margin = settings.rag.min_relevance_margin
# 측정된 제안값과 권장 보수값 중 **더 낮은 쪽**을 쓴다.
# margin 을 높게 잡으면 애매한 근거까지 sufficient 로 놓칠 위험 대신
# 정상 근거를 버려 불필요한 웹 검색이 늘어난다.
settings.rag.min_relevance_margin = min(float(measured_margin), RECOMMENDED_MARGIN)

show_table(
    [
        {"설정": "score_threshold_calibrated", "값": settings.rag.score_threshold_calibrated},
        {"설정": "corpus_score_mean", "값": settings.rag.corpus_score_mean},
        {"설정": "corpus_score_std", "값": settings.rag.corpus_score_std},
        {"설정": "min_relevance_score (절대·보조)", "값": settings.rag.min_relevance_score},
        {"설정": "min_relevance_margin (측정 제안값)", "값": round(float(measured_margin), 4)},
        {"설정": "min_relevance_margin (적용값)", "값": settings.rag.min_relevance_margin},
        {"설정": "min_documents_for_sufficient", "값": settings.rag.min_documents_for_sufficient},
        {"설정": "min_topic_coverage", "값": settings.rag.min_topic_coverage},
    ],
    "calibration 적용 결과",
)

assert settings.rag.score_threshold_calibrated is True, "calibration 이 적용되지 않았습니다."
print("\nscore_threshold_calibrated = True — 이제 충분성 판정을 신뢰할 수 있습니다.")

In [ ]:
# 충분성 평가 재실행 (calibrated 설정 반영) — 명세 14·17절
from petcare_ai.rag.query_builder import build_rag_query
from petcare_ai.rag.sufficiency import KnowledgeSufficiencyEvaluator

evaluator = KnowledgeSufficiencyEvaluator(settings, llm=llm)
rows = []

for question, species in EXAMPLE_QUERIES:
    result = retrieval_results.get(question)
    if result is None:
        continue
    rag_query = build_rag_query(question, profile_for(species), [], [], llm)
    verdict = evaluator.evaluate(rag_query, result.documents)
    rows.append(
        {
            "질문": question[:32],
            "종": rag_query.species,
            "status": verdict.status,
            "Tavily 호출": "필요" if verdict.status in ("insufficient", "conflicting") or verdict.requires_recent_information else "불필요",
            "covered": ", ".join(verdict.covered_topics[:3]),
            "missing": ", ".join(verdict.missing_topics[:3]),
            "최신정보필요": verdict.requires_recent_information,
        }
    )

show_table(rows, "충분성 평가 결과 (명세 18절 산출물 5)")

In [ ]:
# 하드 가드 검증 — LLM 이 sufficient 라 해도 덮어쓸 수 없는 조건 (명세 14절)
from petcare_ai.schemas import RagQuery, RetrievedEvidence

guard_rows = []

# (1) 검색 결과가 0건이면 무조건 insufficient
empty_query = RagQuery(primary_query_ko="테스트", primary_query_en="test", species="dog")
guard_rows.append(
    {
        "케이스": "검색 결과 0건",
        "기대": "insufficient",
        "실제": evaluator.evaluate(empty_query, []).status,
    }
)

# (2) species 불일치 문서만 있으면 insufficient
mismatched = [
    RetrievedEvidence(
        chunk_id="x::0001",
        document_id="x",
        title="Feline lower urinary tract disease",
        text="Cats with urinary obstruction require emergency care.",
        species="cat",
        source="Cornell Feline Health Center",
        source_url="https://www.vet.cornell.edu/feline",
        score=0.62,
    )
]
dog_query = RagQuery(primary_query_ko="강아지 구토", primary_query_en="dog vomiting", species="dog")
guard_rows.append(
    {
        "케이스": "species 불일치 (dog 질문 ← cat 문서)",
        "기대": "insufficient",
        "실제": evaluator.evaluate(dog_query, mismatched).status,
    }
)

show_table(guard_rows, "충분성 하드 가드")

---
## 10. Tavily 수의학 fallback — mock 기본 / 실제 호출은 선택 (명세 15절)

**mock 이 기본입니다.** 실제 Tavily 호출은 `RUN_REAL_TAVILY = True` 로 바꿔야만 실행됩니다.
API 키가 있어도 노트북을 위에서 아래로 돌리는 것만으로 과금되면 안 되기 때문입니다.

검증 규칙(명세 15절):

- allowlist 도메인(`vet.cornell.edu`, `merckvetmanual.com`, `avma.org`, `aspca.org` …)만 accepted
- 커뮤니티·블로그·쇼핑 페이지는 rejected + 사유 기록
- API 오류 / 키 없음 / 결과 없음 = **예외가 아니라 정상 fallback 실패** → 빈 리스트
- 검증을 통과한 근거만 `merge_evidence()` 로 넘어갑니다

In [ ]:
# Tavily client mock — 테스트가 쓰는 것과 같은 형태(호출 인자를 기록한다)
from typing import Any


class MockTavilyClient:
    """Tavily client 대역. `calls` 길이가 곧 '몇 번 호출했는가' 다."""

    def __init__(self, results: list[dict] | None = None, error: Exception | None = None) -> None:
        self.results = list(results or [])
        self.error = error
        self.calls: list[dict[str, Any]] = []

    def search(self, **kwargs: Any) -> dict:
        self.calls.append(kwargs)
        if self.error is not None:
            raise self.error
        return {"results": list(self.results)}


MOCK_VET_RESULTS = [
    {
        "title": "Lily Toxicity in Cats",
        "url": "https://www.vet.cornell.edu/departments-centers-and-institutes/feline-health-center/lily-toxicity",
        "content": "All parts of true lilies are highly toxic to cats and can cause acute kidney injury. Immediate veterinary treatment is required.",
        "score": 0.94,
    },
    {
        "title": "Lily Poisoning - Toxicology",
        "url": "https://www.merckvetmanual.com/toxicology/lily-poisoning",
        "content": "Ingestion of Lilium or Hemerocallis species by cats causes renal tubular necrosis. Decontamination and IV fluids are indicated.",
        "score": 0.91,
    },
    {
        "title": "우리 고양이 백합 먹었어요 후기",
        "url": "https://blog.naver.com/petlove/223311",
        "content": "커뮤니티 후기 글입니다.",
        "score": 0.72,
    },
    {
        "title": "반려묘 해독 영양제 최저가",
        "url": "https://shop.example.com/product/detox-supplement",
        "content": "지금 구매하면 50% 할인.",
        "score": 0.65,
    },
]
print(f"mock 결과 {len(MOCK_VET_RESULTS)}건 준비 (allowlist 2 / 커뮤니티 1 / 쇼핑 1)")

In [ ]:
# mock 으로 수의학 웹 검색 → 출처 검증
from petcare_ai.rag.source_validator import WebSourceValidator, accepted_only
from petcare_ai.rag.tavily_vet_search import VeterinaryWebSearchService

mock_client = MockTavilyClient(MOCK_VET_RESULTS)
mock_web_search = VeterinaryWebSearchService(settings, client=mock_client)
validator = WebSourceValidator(settings)

raw_items = mock_web_search.search("cat lily ingestion toxicity", "cat", max_results=5)
validated = validator.validate(raw_items, "cat")
accepted = accepted_only(validated)

show_table(
    [
        {
            "제목": item.title[:34],
            "domain": item.domain,
            "accepted": item.accepted,
            "reject_reason": item.reject_reason[:46],
        }
        for item in validated
    ],
    "Tavily mock 결과 검증 (명세 18절 산출물 6)",
)
print(f"\nTavily 호출 횟수: {len(mock_client.calls)}")
print(f"수집 {len(raw_items)}건 → accepted {len(accepted)}건 / rejected {len(validated) - len(accepted)}건")
print("include_domains 가 전달됐는가:", bool(mock_client.calls and mock_client.calls[0].get("include_domains")))

In [ ]:
# API 오류 시 안전하게 실패하는지 (명세 17절 Tavily 테스트)
failing = VeterinaryWebSearchService(settings, client=MockTavilyClient(error=RuntimeError("429 rate limit")))
try:
    fallback_items = failing.search("cat lily toxicity", "cat")
    print(f"오류 발생 시 반환값: {fallback_items} → 예외를 던지지 않고 빈 리스트 (정상 fallback)")
except Exception as exc:
    print(f"[문제] 예외가 밖으로 나왔습니다: {exc}")

# 조건부 호출 검증 — sufficient 인 질문에서는 Tavily 를 부르지 않는다 (명세 47절)
from petcare_ai.rag.service import VeterinaryRagService

counting_client = MockTavilyClient(MOCK_VET_RESULTS)
rag_service = VeterinaryRagService(
    settings=settings,
    store=service.store,                                  # 7번에서 만든 index 재사용
    llm=llm,
    web_search=VeterinaryWebSearchService(settings, client=counting_client),
    validator=validator,
)

rows = []
for question, species in EXAMPLE_QUERIES:
    before = len(counting_client.calls)
    result, merged = rag_service.retrieve_with_fallback(question, profile_for(species))
    rows.append(
        {
            "질문": question[:32],
            "충분성": result.sufficiency,
            "web_fallback_required": result.web_fallback_required,
            "Tavily 실제 호출": len(counting_client.calls) - before,
            "최종 근거": len(merged.evidence),
            "web 근거": sum(1 for e in merged.evidence if e.source_type == "web"),
            "신뢰 근거": merged.has_reliable_evidence,
        }
    )

show_table(rows, "조건부 Tavily 호출 검증 (sufficient → 호출 0)")
print(f"\n총 Tavily 호출: {len(counting_client.calls)}회 (모든 질문에 호출했다면 {len(EXAMPLE_QUERIES)}회여야 함)")

In [ ]:
# 선택적 실제 Tavily 호출 — 기본값 False (명세 15절)
RUN_REAL_TAVILY = False

if not RUN_REAL_TAVILY:
    print("RUN_REAL_TAVILY=False — 실제 Tavily 호출을 건너뜁니다. (mock 결과만 사용)")
elif not os.environ.get("TAVILY_API_KEY"):
    print("TAVILY_API_KEY 가 없어 실제 호출을 할 수 없습니다.")
else:
    real_search = VeterinaryWebSearchService(settings)
    if not real_search.is_available():
        print("tavily-python 미설치 또는 client 생성 실패 — 건너뜁니다.")
    else:
        real_items = real_search.search("cat lily ingestion acute kidney injury", "cat", max_results=5)
        real_validated = validator.validate(real_items, "cat")
        show_table(
            [
                {
                    "제목": item.title[:34],
                    "domain": item.domain,
                    "accepted": item.accepted,
                    "reject_reason": item.reject_reason[:40],
                }
                for item in real_validated
            ],
            "실제 Tavily 호출 결과",
        )

---
## 11. RAG 전체 테스트 (pytest)

RAG 계층 테스트 4개 파일을 실행합니다.

**예상 결과: `203 passed`**

| 파일 | 검증 범위 |
| --- | --- |
| `test_rag_loader.py` | 문서 로드·검증·중복 제거·categories 합집합 병합 |
| `test_rag_retrieval.py` | chunking / FAISS 저장·로드·지문 / ko+en 검색 / 종 분리 |
| `test_rag_sufficiency.py` | 충분성 판정·하드 가드·calibration |
| `test_tavily_fallback.py` | 조건부 호출·allowlist 검증·오류 흡수·evidence merge |

In [ ]:
import os
import sys

os.chdir(PROJECT_ROOT)
print("cwd        :", os.getcwd())
print("python     :", sys.executable)
print("테스트 대상: petcare_ai/tests/test_rag_*.py + test_tavily_fallback.py")

In [ ]:
!python -m pytest petcare_ai/tests/test_rag_*.py petcare_ai/tests/test_tavily_fallback.py -q

### 명세 18절 RAG 완료 산출물 체크

| # | 산출물 | 확인 위치 |
| --- | --- | --- |
| 1 | 데이터 통계 | 5번 섹션 |
| 2 | chunk 통계 | 6번 섹션 |
| 3 | FAISS index 저장 경로 | 7번 섹션 |
| 4 | query별 검색 결과 표 | 8번 섹션 |
| 5 | 충분성 평가 결과 | 9번 섹션 |
| 6 | Tavily fallback mock 결과 | 10번 섹션 |
| 7 | pytest 결과 | 11번 섹션 (203 passed) |

**여기까지 통과해야 LangGraph 로 넘어갑니다**(명세 5·47절: RAG 테스트 전에 LangGraph 를 작성하지 않는다).

---
## 12. 기존 일기장·진단서 adapter 연결 (명세 4절)

**일기장·진단서 처리 로직을 재작성하지 않습니다.** `ClinicalDataAdapter` 라는 단일 통로만 두고,
Colab 에서는 서버 코드가 없으므로 명세 43절 시나리오를 그대로 담은 `FixtureClinicalDataAdapter` 를 씁니다.

- `USE_EXISTING_PROCESSORS=True` 인데 기존 모듈 import 에 실패하면
  **조용히 fixture 로 넘어가지 않고 `RuntimeError`** 를 던집니다(무엇을 쓰고 있는지 늘 분명해야 하므로).
- 서버 코드가 있는 환경이라면 `PETCARE_SERVER_PATH` 환경 변수로 경로를 지정하세요.

### fixture 데이터 (명세 43절)

| pet_id | 이름 | 종 | 특징 |
| --- | --- | --- | --- |
| 1 | 초코 | dog / poodle / 8세 | 이첨판 폐쇄부전증(ACVIM B2), 심장약 복용, 최근 식사·활동 감소 |
| 2 | 나비 | cat / korean shorthair / 3세 | 기저질환 없음 → 종 분리 검증용 |

**체중 충돌 fixture**: PET DB **5.2kg** vs 최신 진단서 **4.8kg** (+ 대화 중 사용자 추정 4.5kg → 3중 충돌)

In [ ]:
from petcare_ai.adapters.clinical_data_adapter import (
    USE_EXISTING_PROCESSORS,
    FixtureClinicalDataAdapter,
    get_adapter,
    load_daily_entries_for_test,
    load_diagnoses_for_test,
    load_pet_profile_for_test,
)

USE_EXISTING = False   # Colab 에는 서버 코드가 없으므로 fixture 를 씁니다.

try:
    adapter = get_adapter(use_existing=USE_EXISTING)
except RuntimeError as exc:
    print("기존 처리 코드 import 실패 — fixture 로 진행합니다.")
    print(" 사유:", exc)
    adapter = FixtureClinicalDataAdapter()

print("모듈 기본값 USE_EXISTING_PROCESSORS =", USE_EXISTING_PROCESSORS)
print("이 노트북에서 사용하는 adapter :", type(adapter).__name__)

In [ ]:
# fixture 내용 확인
import json

for pet_id in (1, 2):
    profile = adapter.load_pet_profile(pet_id)
    diagnoses = adapter.load_diagnoses(pet_id)
    entries = adapter.load_daily_entries(pet_id)
    print(f"\n=== pet_id={pet_id} / {profile.get('name')} ({profile.get('species')}) ===")
    print(f"  품종 {profile.get('breed')} / 나이 {profile.get('age_years')}세 / 체중 {profile.get('weight_kg')}kg")
    print(f"  기저질환: {profile.get('diseases') or '없음'}")
    print(f"  복용약  : {profile.get('medications') or '없음'}")
    print(f"  진단서 {len(diagnoses)}건 / 일기 {len(entries)}건")
    if diagnoses:
        latest = diagnoses[-1]
        print(f"  최신 진단서({latest.get('date')}): {latest.get('diagnosis')}")

# 명세 43절 체중 충돌 fixture 확인
pet1 = adapter.load_pet_profile(1)
latest_diag = adapter.load_diagnoses(1)[-1]
show_table(
    [
        {"출처": "PET DB", "체중": f"{pet1.get('weight_kg')}kg", "비고": "profile.weight_kg"},
        {"출처": "최신 진단서", "체중": "4.8kg", "비고": latest_diag.get("date")},
        {"출처": "사용자 현재 입력", "체중": "4.5kg", "비고": "21번 섹션 시나리오에서 대화로 전달"},
    ],
    "명세 43절 Context 충돌 fixture",
)

# 명세 4절이 요구한 이름도 그대로 제공된다.
print("\nload_pet_profile_for_test(1)['name'] =", load_pet_profile_for_test(1)["name"])
print("load_diagnoses_for_test(1) 건수      =", len(load_diagnoses_for_test(1)))
print("load_daily_entries_for_test(1) 건수  =", len(load_daily_entries_for_test(1)))

---
## 13. LangGraph State 및 node (명세 25·23절)

`PetCareState` 가 graph 하위 모든 node 가 읽고 쓰는 **유일한 계약**입니다.

reducer 설계:

- **누적 + 중복 제거**: `red_flags`, `risk_reasons`, `missing_fields`, `ui_actions`, `merged_evidence`
  → 병렬 node(Rule Assessment ∥ Assessment Agent)가 같은 key 를 써도 충돌하지 않고,
    뒤 node 가 앞 node 의 red flag 를 지울 수 없습니다.
- **상향 전용(escalate)**: `final_risk`, `emergency_urgency`
  → 명세 28절 "낮은 위험도로 덮어쓰지 않는다" 를 타입 수준에서 강제합니다.
- 리스트를 줄여야 하는 정상 상황(41절 대화 요약)을 위해 `Replace` / `ReplaceDict` 탈출구를 둡니다.

In [ ]:
from petcare_ai.graph.state import (
    URGENCY_PRIORITY,
    PetCareState,
    build_trace_metadata,
    escalate_risk,
    escalate_urgency,
    make_initial_state,
)

demo_state = make_initial_state(
    pet_id=1,
    user_message="강아지가 오늘 한 번 토했는데 지금은 잘 놀아요",
    thread_id="thread-demo",
    region_name="서울 강남구",
    pet_profile=adapter.load_pet_profile(1),
    diagnoses=adapter.load_diagnoses(1),
    daily_entries=adapter.load_daily_entries(1),
)

print("State key 수:", len(demo_state))
show_table(
    [
        {"key": k, "값": str(demo_state[k])[:56]}
        for k in (
            "thread_id", "pet_id", "user_message", "context_loaded", "species",
            "region_name", "final_risk", "emergency_urgency", "rag_sufficiency",
            "web_fallback_required", "retry_count",
        )
        if k in demo_state
    ],
    "초기 State 주요 필드",
)

# 상향 전용 reducer 확인 (명세 28절)
print("\nescalate_risk('emergency','normal') =", escalate_risk("emergency", "normal"), "← 낮은 값으로 못 내린다")
print("escalate_risk('normal','visit')     =", escalate_risk("normal", "visit"))
print("escalate_urgency 우선순위           =", URGENCY_PRIORITY)

In [ ]:
# 개별 node 를 직접 호출해 동작을 확인합니다(그래프 없이도 순수 함수로 검증 가능).
from petcare_ai.graph.nodes.fast_emergency_guard import fast_emergency_guard_node
from petcare_ai.graph.nodes.supervisor import make_supervisor_node

supervisor_node = make_supervisor_node(llm)
rows = []

for message in (
    "안녕",
    "이 앱에서 무엇을 할 수 있어?",
    "강아지가 오늘 한 번 토했는데 지금은 잘 놀아요",
    "지금 계속 경련하고 반응이 둔해요",
    "숨을 거의 쉬지 못하고 의식이 없어요",
):
    state = dict(demo_state)
    state["user_message"] = message
    guard = fast_emergency_guard_node(state) or {}
    merged = {**state, **guard}
    sup = supervisor_node(merged) or {}
    rows.append(
        {
            "메시지": message[:30],
            "fast guard risk": guard.get("final_risk", "-"),
            "urgency": guard.get("emergency_urgency", "-"),
            "red_flags": len(guard.get("red_flags") or []),
            "intent": sup.get("intent", "-"),
            "possible_emergency": sup.get("possible_emergency", "-"),
        }
    )

show_table(rows, "Fast Emergency Guard + Supervisor 동작")

---
## 14. Health Subgraph (명세 30절)

일반 건강 상담 경로입니다. RAG 검색 → 충분성 판정 → (부족할 때만) 웹 fallback →
근거 병합 → 답변 생성 순서로 돕니다.

서브그래프는 **부모와 같은 `PetCareState`** 를 쓰므로 상태 변환 래퍼가 필요 없고,
compile 된 그래프를 그대로 부모의 node 로 끼울 수 있습니다.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

from petcare_ai.graph.subgraphs import SubgraphDeps, build_health_subgraph

# 서브그래프를 단독 실행하려면 자체 checkpointer 가 필요합니다(interrupt 대비).
# 부모 그래프에 끼울 때는 checkpointer 를 주지 않고 부모 것을 상속합니다.
health_deps = SubgraphDeps(
    settings=settings,
    llm=llm,
    rag_service=rag_service,     # 10번에서 mock Tavily 를 물린 서비스
    checkpointer=InMemorySaver(),
)
health_graph = build_health_subgraph(health_deps)
print("Health Subgraph node:", sorted(health_graph.get_graph().nodes))

In [ ]:
health_state = make_initial_state(
    pet_id=1,
    user_message="강아지가 오늘 한 번 토했는데 지금은 잘 놀아요",
    thread_id="thread-health-demo",
    pet_profile=adapter.load_pet_profile(1),
    diagnoses=adapter.load_diagnoses(1),
    daily_entries=adapter.load_daily_entries(1),
)
health_config = {"configurable": {"thread_id": "thread-health-demo"}}

try:
    health_out = health_graph.invoke(health_state, config=health_config)
except Exception as exc:
    health_out = {}
    print(f"[오류] Health Subgraph 실행 실패: {type(exc).__name__}: {exc}")

if health_out:
    show_table(
        [
            {"항목": "rag_sufficiency", "값": health_out.get("rag_sufficiency")},
            {"항목": "rag 문서 수", "값": len(health_out.get("rag_documents") or [])},
            {"항목": "web_fallback_required", "값": health_out.get("web_fallback_required")},
            {"항목": "최종 근거 수", "값": len(health_out.get("merged_evidence") or [])},
            {"항목": "has_reliable_evidence", "값": health_out.get("has_reliable_evidence")},
            {"항목": "final_risk", "값": health_out.get("final_risk")},
        ],
        "Health Subgraph 결과",
    )
    print("\n--- 생성된 답변 ---")
    print((health_out.get("draft_response") or "(없음)")[:900])

---
## 15. Visit Subgraph (명세 31절)

병원 상담 권고 경로입니다. **Missing Information → interrupt → resume** 루프를 거쳐
정보를 모은 뒤 `ConsultationPacket` → PDF → EmailDraft 를 만듭니다.

되묻기가 무한 반복되지 않도록 라운드 수를 세고 상한에서 빠져나옵니다
(부족한 항목은 지우지 않고 PDF `unknown_fields` 로 남깁니다 — 추측해 채우지 않습니다).

In [ ]:
from petcare_ai.graph.subgraphs import build_visit_subgraph

visit_deps = SubgraphDeps(
    settings=settings,
    llm=llm,
    rag_service=rag_service,
    checkpointer=InMemorySaver(),
)
visit_graph = build_visit_subgraph(visit_deps)
print("Visit Subgraph node:", sorted(visit_graph.get_graph().nodes))

visit_state = make_initial_state(
    pet_id=1,
    user_message="며칠째 식사를 거의 안 하고 활동량이 계속 줄었어요",
    thread_id="thread-visit-sub",
    pet_profile=adapter.load_pet_profile(1),
    diagnoses=adapter.load_diagnoses(1),
    daily_entries=adapter.load_daily_entries(1),
    final_risk="visit",
)
visit_config = {"configurable": {"thread_id": "thread-visit-sub"}}

try:
    visit_out = visit_graph.invoke(visit_state, config=visit_config)
    snapshot = visit_graph.get_state(visit_config)
    pending = [
        getattr(interrupt, "value", None)
        for task in (getattr(snapshot, "tasks", ()) or ())
        for interrupt in (getattr(task, "interrupts", ()) or ())
    ]
    print("\nmissing_fields :", visit_out.get("missing_fields"))
    print("대기 중 interrupt:", pending[:1] if pending else "없음")
except Exception as exc:
    print(f"[오류] Visit Subgraph 실행 실패: {type(exc).__name__}: {exc}")

---
## 16. Emergency Subgraph (명세 32절)

응급 경로입니다. 즉시 안내 메시지를 먼저 내보내고,
지역 확인 → 병원 요구사항 생성(PET DB + 진단서 기준) → 병원 검색 → 적합도 평가 →
PDF/EmailDraft → `CALL_HOSPITAL` 등 `ui_actions` 생성 순으로 진행합니다.

- 병원 검색은 **수의학 지식 검색과 별도 class**(`HospitalSearchService`)입니다.
- 검색 결과로 **실시간 진료 가능 여부를 확정하지 않습니다** — 모든 후보에 `전화 확인 필요` 를 붙입니다.
- 여기서는 Tavily 키 없이도 되도록 **mock 병원 검색**을 주입합니다.

In [ ]:
# 명세 43절 '병원 적합도' 4개 후보를 mock 으로 준비합니다.
from petcare_ai.rag.tavily_vet_search import HospitalSearchService

MOCK_HOSPITAL_RESULTS = [
    {
        "title": "행복동물병원 | 강남구 일반 진료",
        "url": "https://happy-animal.example.kr",
        "content": "강남구 소재 동물병원입니다. 내과·외과 일반 진료. 전화 02-555-1234. 진료시간 09:00-19:00.",
        "score": 0.82,
    },
    {
        "title": "24시 강남응급동물의료센터",
        "url": "https://gangnam-emergency-vet.example.kr",
        "content": "24시간 연중무휴 응급 진료 동물병원. 응급실 운영, 심장 전문 진료 가능. 전화 02-777-0119.",
        "score": 0.93,
    },
    {
        "title": "서울동물메디컬센터 - 심장내과",
        "url": "https://seoul-amc.example.kr",
        "content": "심장 초음파·순환기 전문 진료 동물병원. 이첨판 폐쇄부전증 관리. 전화 02-333-8282.",
        "score": 0.88,
    },
    {
        "title": "라온동물의료센터 심장 전문",
        "url": "https://raon-vet.example.kr",
        "content": "심장 전문 진료와 응급 대응이 가능한 동물병원입니다. 연락처는 홈페이지 참고.",
        "score": 0.74,
    },
]

mock_hospital_search = HospitalSearchService(settings, client=MockTavilyClient(MOCK_HOSPITAL_RESULTS))
print("mock 병원 후보:", len(MOCK_HOSPITAL_RESULTS), "건 (일반 / 24시응급 / 기존진료+전문 / 전화번호 없음)")

In [ ]:
from petcare_ai.graph.subgraphs import build_emergency_subgraph

emergency_deps = SubgraphDeps(
    settings=settings,
    llm=llm,
    rag_service=rag_service,
    hospital_search=mock_hospital_search,
    checkpointer=InMemorySaver(),
)
emergency_graph = build_emergency_subgraph(emergency_deps)
print("Emergency Subgraph node:", sorted(emergency_graph.get_graph().nodes))

emergency_state = make_initial_state(
    pet_id=1,
    user_message="지금 계속 경련하고 반응이 둔해요",
    thread_id="thread-emergency-sub",
    region_name="서울 강남구",     # Android 위치 입력 대신 쓰는 fixture (명세 32절)
    pet_profile=adapter.load_pet_profile(1),
    diagnoses=adapter.load_diagnoses(1),
    daily_entries=adapter.load_daily_entries(1),
    final_risk="emergency",
    emergency_urgency="contact_ready",
)
emergency_config = {"configurable": {"thread_id": "thread-emergency-sub"}}

try:
    emergency_out = emergency_graph.invoke(emergency_state, config=emergency_config)
except Exception as exc:
    emergency_out = {}
    print(f"[오류] Emergency Subgraph 실행 실패: {type(exc).__name__}: {exc}")

if emergency_out:
    show_table(
        [
            {"항목": "emergency_urgency", "값": emergency_out.get("emergency_urgency")},
            {"항목": "region_name", "값": emergency_out.get("region_name")},
            {"항목": "검색어 수", "값": len(emergency_out.get("hospital_search_queries") or [])},
            {"항목": "병원 후보(원시)", "값": len(emergency_out.get("raw_hospital_results") or [])},
            {"항목": "병원 평가 결과", "값": len(emergency_out.get("hospital_results") or [])},
            {"항목": "pdf_path", "값": str(emergency_out.get("pdf_path"))[-52:]},
            {"항목": "email_draft", "값": bool(emergency_out.get("email_draft"))},
            {"항목": "ui_actions", "값": [a.get("type") for a in (emergency_out.get("ui_actions") or [])]},
        ],
        "Emergency Subgraph 결과",
    )
    print("\n--- 응급 안내 ---")
    print((emergency_out.get("draft_response") or "(없음)")[:900])

---
## 17. Main Graph compile (명세 24절)

```
START → Context loaded? → DB Context → Message Ingest
      → Need summary? → Summary → Fast Emergency Guard
      → Critical? → Emergency Subgraph / Supervisor
      → Intent 분기(General / Health / Hospital Search / Unsupported)
      → Health 경로: Clinical Context Priority
                   → Rule Assessment ∥ Assessment Agent
                   → Risk Double Check → Merge Risk
      → Final Risk 분기(Normal→Health / Visit→Visit / Emergency→Emergency)
      → 전부 Output Check → Valid?(accept→Final Safety / regenerate→1회 재시도 / fallback→Safe Fallback)
      → Build Result → END
```

`GraphDependencies` 하나만 바꿔 끼우면 **Colab fixture → FastAPI/SQLite** 로 이동합니다.

In [ ]:
from petcare_ai.graph.builder import (
    GraphDependencies,
    build_petcare_graph,
    describe_graph,
    run_chat,
)

checkpointer = InMemorySaver()   # Colab 용. 운영에서는 영속 checkpointer 로 교체(22번 섹션)

deps = GraphDependencies(
    settings=settings,
    llm=llm,                          # None 이어도 규칙 기반으로 끝까지 동작
    rag_service=rag_service,          # mock Tavily 가 물린 RAG 서비스
    hospital_search=mock_hospital_search,
    clinical_adapter=adapter,         # Colab fixture
)

graph = build_petcare_graph(deps, checkpointer=checkpointer)
print("compile 완료.")
print("node:", sorted(graph.get_graph().nodes))

In [ ]:
print(describe_graph(graph))

In [ ]:
# Mermaid 다이어그램 (렌더 실패해도 셀이 죽지 않습니다)
try:
    from IPython.display import Image, display

    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print(f"PNG 렌더 불가({type(exc).__name__}) — Mermaid 텍스트로 대체합니다.\n")
    try:
        print(graph.get_graph().draw_mermaid())
    except Exception as inner:
        print("Mermaid 텍스트도 만들 수 없습니다:", inner)

---
## 18. LangSmith 설정 (명세 42절)

**민감한 원문·개인 식별정보는 metadata 에 넣지 않습니다.**
사용자 메시지, 반려동물 이름, 주소, 전화번호는 제외하고 **분기 결과와 개수만** 남깁니다.

Trace 에서 확인할 것: Supervisor route / Context priority / Rule·Assessment·Double Check /
RAG 문서와 score / Tavily fallback 이유 / 병원 적합도 근거 / Missing Information interrupt /
PDF 생성 성공 여부 / Output Check 실패 이유 / node별 latency·token.

In [ ]:
import os

LANGSMITH_PROJECT = "petcare-colab"

if os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    os.environ.setdefault("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
    print(f"LangSmith tracing 활성화 — project={LANGSMITH_PROJECT}")
    print("확인: https://smith.langchain.com/ → Projects → " + LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LANGSMITH_API_KEY 가 없어 tracing 을 끕니다. (그래프 실행에는 영향 없음)")

# trace metadata 예시 — 원문 없이 범주형 값과 개수만 담긴다.
sample_meta = build_trace_metadata(demo_state, environment=settings.environment)
show_table([{"key": k, "값": str(v)} for k, v in sample_meta.items()], "LangSmith metadata (민감정보 제외)")

SENSITIVE = ("user_message", "name", "phone", "address", "email", "diagnosis")
leaked = [k for k in sample_meta if any(word in k.lower() for word in SENSITIVE)]
print("\n민감 필드 누출 검사:", "없음 (정상)" if not leaked else f"확인 필요 → {leaked}")

# 실행 시 metadata 를 붙이는 방법
RUN_CONFIG_EXAMPLE = {
    "configurable": {"thread_id": "thread-example"},
    "metadata": {"environment": "colab", "scenario": "general_health"},
    "tags": ["petcare", "colab"],
}
print("\nrun_chat(graph, ..., config=RUN_CONFIG_EXAMPLE) 형태로 trace metadata 를 붙일 수 있습니다.")

---
## 19. multi-turn interrupt / resume (명세 29·43절)

`InMemorySaver` + **같은 `thread_id`** + `Command(resume=...)` 조합입니다.
`run_chat(graph, resume=...)` 이 내부에서 `Command(resume=...)` 를 만들어 넣습니다.

- 1턴: 정보가 부족하면 `interrupt` 로 멈추고 질문을 돌려줍니다.
- 2턴: **같은 thread_id** 로 `resume` 에 사용자 답변을 넣으면 멈춘 지점부터 이어집니다.
- pet 별로 thread 를 분리해야 다른 아이의 대화가 섞이지 않습니다.

In [ ]:
VISIT_THREAD = "pet-1-visit-demo"


def final_state_of(thread_id: str) -> dict:
    """checkpointer 에 저장된 최종 State 를 읽는다(실패해도 빈 dict)."""
    try:
        snapshot = graph.get_state({"configurable": {"thread_id": thread_id}})
        return dict(snapshot.values or {})
    except Exception:
        return {}


# --- turn 1: 정보 부족 → interrupt ---
turn1 = run_chat(
    graph,
    pet_id=1,
    user_message="며칠째 식사를 거의 안 하고 활동량이 계속 줄었어요",
    thread_id=VISIT_THREAD,
    region_name="서울 강남구",
)

print("=== turn 1 ===")
print("risk_level        :", turn1.risk_level)
print("interrupted       :", turn1.trace_metadata.get("interrupted"))
print("missing_information:", turn1.missing_information)
print("\n[AI]", (turn1.message or "(없음)")[:600])

In [ ]:
# --- turn 2: 같은 thread_id 로 resume ---
turn2 = run_chat(
    graph,
    thread_id=VISIT_THREAD,      # 반드시 같은 thread_id
    resume="3일 전부터요. 사료를 하루 두 번 주는데 거의 안 먹고, 물만 조금 마셔요. 구토는 없어요.",
)

print("=== turn 2 (resume) ===")
print("risk_level  :", turn2.risk_level)
print("interrupted :", turn2.trace_metadata.get("interrupted", False))
print("pdf_path    :", turn2.pdf_path)
print("email_draft :", bool(turn2.email_draft))
print("ui_actions  :", [a.get("type") for a in (turn2.ui_actions or [])])
print("근거 수     :", len(turn2.evidence))
print("\n[AI]", (turn2.message or "(없음)")[:900])

if turn2.email_draft:
    print("\n--- Email Draft (실제 전송하지 않습니다) ---")
    print("to      :", turn2.email_draft.to)
    print("subject :", turn2.email_draft.subject)
    print("첨부    :", turn2.email_draft.attachment_filename)
    print(turn2.email_draft.body[:600])

In [ ]:
# State 유지 확인 — 같은 thread 에서 값이 이어지는가 / pet 별 thread 가 섞이지 않는가
visit_state_final = final_state_of(VISIT_THREAD)

show_table(
    [
        {"항목": "thread_id", "값": visit_state_final.get("thread_id")},
        {"항목": "pet_id", "값": visit_state_final.get("pet_id")},
        {"항목": "누적 메시지 수", "값": len(visit_state_final.get("messages") or [])},
        {"항목": "context_loaded", "값": visit_state_final.get("context_loaded")},
        {"항목": "collected_information", "값": str(visit_state_final.get("collected_information"))[:70]},
        {"항목": "final_risk", "값": visit_state_final.get("final_risk")},
    ],
    "multi-turn State 유지",
)

# pet 2 는 다른 thread — 초코 정보가 섞이면 안 된다.
other = run_chat(
    graph,
    pet_id=2,
    user_message="고양이가 오늘 평소보다 조금 덜 먹었어요",
    thread_id="pet-2-demo",
)
other_state = final_state_of("pet-2-demo")
print("\npet-2 thread 의 pet_id :", other_state.get("pet_id"))
print("pet-2 thread 의 이름   :", (other_state.get("pet_profile") or {}).get("name"))
print("thread 격리:", "정상" if other_state.get("pet_id") == 2 else "확인 필요")

---
## 20. PDF 생성 (명세 36·37절)

`build_consultation_packet(state)` 는 순수 함수입니다 — LLM 을 쓰지 않고
Clinical Context Priority 가 **이미 선택해 둔** 진단서·일기 레코드를 원문 그대로 싣습니다.

- **누락 필드를 추측해 채우지 않습니다.** 없으면 `unknown_fields` 에 남기고 PDF 에 `미확인` 으로 인쇄합니다.
- 모든 페이지 하단에 "확정 진단이 아니며 수의사 확인이 필요합니다" 고지가 들어갑니다.
- 한글 폰트가 없으면 글자가 깨지므로 Colab 에서는 나눔폰트 설치를 권장합니다.

In [ ]:
# Colab 한글 폰트 — 없으면 PDF 의 한글이 깨집니다.
from petcare_ai.pdf.consultation_pdf import find_korean_font

font_path = find_korean_font()
print("탐지된 한글 폰트:", font_path or "없음")

if not font_path and IN_COLAB:
    print("\n나눔폰트를 설치하려면 아래 셀을 실행한 뒤 이 셀을 다시 실행하세요:")
    print("  !apt-get -qq install -y fonts-nanum")

In [ ]:
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1 ; echo "fonts-nanum 설치 시도 완료 (Colab 전용)"

In [ ]:
# 19번 turn 2 의 최종 State 로 상담 packet → PDF 생성
from petcare_ai.graph.nodes.document_agent import build_consultation_packet
from petcare_ai.pdf.consultation_pdf import generate_consultation_pdf

pdf_state = final_state_of(VISIT_THREAD)

if not pdf_state:
    print("[안내] 19번 섹션을 먼저 실행하세요. State 를 읽지 못했습니다.")
else:
    pdf_state.setdefault("document_type", "visit_consultation")
    packet = build_consultation_packet(pdf_state)

    show_table(
        [
            {"항목": "document_type", "값": packet.document_type},
            {"항목": "generated_at", "값": packet.generated_at},
            {"항목": "pet 필드 수", "값": len(packet.pet)},
            {"항목": "관련 진단서", "값": len(packet.related_diagnoses)},
            {"항목": "관련 일기", "값": len(packet.supporting_daily_entries)},
            {"항목": "unknown_fields (추측 금지)", "값": ", ".join(packet.unknown_fields) or "없음"},
            {"항목": "provenance 기록", "값": len(packet.provenance)},
        ],
        "ConsultationPacket",
    )

    try:
        pdf_path, pdf_filename = generate_consultation_pdf(packet, output_dir=settings.output_dir)
        size = Path(pdf_path).stat().st_size
        print(f"\nPDF 생성 완료: {pdf_filename} ({size:,} bytes)")
        print("경로:", pdf_path)
        assert size > 0, "PDF 크기가 0 입니다."
    except Exception as exc:
        pdf_path = None
        print(f"\n[오류] PDF 생성 실패: {type(exc).__name__}: {exc}")

In [ ]:
# 생성된 PDF 확인 / 다운로드
if 'pdf_path' in dir() and pdf_path:
    try:
        from IPython.display import IFrame, display

        display(IFrame(pdf_path, width=760, height=520))
    except Exception:
        pass
    if IN_COLAB:
        try:
            from google.colab import files  # type: ignore[import-not-found]

            # files.download(pdf_path)   # ← 주석 해제하면 내려받습니다.
            print("다운로드하려면 위 files.download(pdf_path) 주석을 해제하세요.")
        except Exception as exc:
            print("다운로드 위젯 사용 불가:", exc)
else:
    print("생성된 PDF 가 없습니다.")

print("\noutputs 디렉터리:")
for item in sorted(Path(settings.output_dir).glob("*.pdf"))[-5:]:
    print(f"  {item.name}  ({item.stat().st_size:,} bytes)")

---
## 21. 최종 시나리오 회귀 테스트 (명세 43절 전체)

| # | 시나리오 | 기대 |
| --- | --- | --- |
| 1 | 일반 대화 | `general_chat` 분기 / RAG·Tavily 미호출 |
| 2 | 일반 건강 상담 | `normal` 가능 / RAG 호출 / 충분하면 Tavily 미호출 |
| 3 | RAG fallback | `insufficient` → Tavily 호출 / 검증된 출처만 evidence |
| 4 | 병원 상담 권고 | `visit` 분기 / interrupt → resume → PDF + email draft |
| 5 | 응급 | `emergency` / 병원 검색 / 요구사항 / 적합도 / PDF·email |
| 6 | 즉시 위급 | `critical_immediate` / 정보 부족해도 `CALL_HOSPITAL` |
| 7 | 종 분리 | cat index 만 사용 / dog 문서 미포함 |
| 8 | Context 충돌 | PET 5.2 vs 진단서 4.8 vs 사용자 4.5 → 충돌 기록·PDF 보존 |
| 9 | 병원 적합도 | 후보별 score 와 이유 |
| 10 | Output Safety | 확정 진단 / 약 중단 지시 / "현재 진료 가능" 단정 차단 |

In [ ]:
# 시나리오 실행 헬퍼
import uuid

regression_rows = []


def record(no, name, expected, actual, ok, note=""):
    regression_rows.append(
        {
            "#": no,
            "시나리오": name,
            "기대": expected,
            "실제": actual,
            "판정": "PASS" if ok else "CHECK",
            "비고": note[:70],
        }
    )


def run_scenario(pet_id, message, thread_id=None, region_name=None, **kwargs):
    """graph 를 1턴 실행하고 (결과, 최종 State) 를 돌려준다. 예외는 흡수한다."""
    thread_id = thread_id or f"reg-{uuid.uuid4().hex[:8]}"
    try:
        result = run_chat(
            graph,
            pet_id=pet_id,
            user_message=message,
            thread_id=thread_id,
            region_name=region_name,
            **kwargs,
        )
        return result, final_state_of(thread_id), thread_id, None
    except Exception as exc:
        return None, final_state_of(thread_id), thread_id, exc


print("헬퍼 준비 완료.")

In [ ]:
# 1. 일반 대화 — RAG / Tavily 미호출
calls_before = len(counting_client.calls)
res, state, tid, err = run_scenario(1, "안녕")
res2, state2, tid2, err2 = run_scenario(1, "이 앱에서 무엇을 할 수 있어?")

intent = state.get("intent")
rag_used = bool(state.get("rag_documents")) or bool(state2.get("rag_documents"))
tavily_used = len(counting_client.calls) - calls_before

record(
    1, "일반 대화", "general_chat / RAG·Tavily 미호출",
    f"intent={intent} / RAG={'호출' if rag_used else '미호출'} / Tavily={tavily_used}회",
    intent == "general_chat" and not rag_used and tavily_used == 0,
    (res.message if res else str(err))[:60],
)
print("[AI]", (res2.message if res2 else str(err2))[:400])

In [ ]:
# 2. 일반 건강 상담 — normal 가능 / RAG 호출 / 충분하면 Tavily 미호출
HEALTH_MESSAGE = "강아지가 오늘 한 번 토했는데 지금은 잘 놀아요"

calls_before = len(counting_client.calls)
res, state, tid, err = run_scenario(1, HEALTH_MESSAGE)

rag_count = len(state.get("rag_documents") or [])
tavily_used = len(counting_client.calls) - calls_before
risk = state.get("final_risk")

# 초코는 심장질환 병력이 있어 risk 가 visit 로 올라갈 수 있습니다(정상 — 명세는 "normal 가능").
# 그 경우 health subgraph 를 지나지 않으므로, RAG 자체가 도는지는 계층에서 직접 확인합니다.
direct = rag_service.retrieve(HEALTH_MESSAGE, profile_for("dog"))

record(
    2, "일반 건강 상담", "normal/visit 허용 / RAG 호출",
    f"risk={risk} / 그래프 RAG={rag_count}건 / RAG계층={len(direct.documents)}건 / "
    f"sufficiency={direct.sufficiency} / Tavily={tavily_used}회",
    res is not None and (rag_count > 0 or len(direct.documents) > 0),
    f"그래프 충분성={state.get('rag_sufficiency')}",
)
print("[AI]", (res.message if res else str(err))[:500])

In [ ]:
# 3. RAG fallback — insufficient → Tavily 호출 → 검증된 출처만 evidence
#
# 그래프는 이 질문을 위험도에 따라 visit/emergency 로 보낼 수 있어 web fallback 노드를
# 지나지 않을 수 있습니다. 그래서 fallback 조건 자체는 **RAG 서비스 계층에서 직접**
# 검증하고(결정적), 그래프 실행 결과는 참고로 함께 보여 줍니다.
calls_before = len(counting_client.calls)
rag_result, rag_merged = rag_service.retrieve_with_fallback(
    "고양이가 백합을 먹은 것 같아요", profile_for("cat")
)
service_tavily = len(counting_client.calls) - calls_before
service_web = [e for e in rag_merged.evidence if e.source_type == "web"]
print(
    f"[RAG 계층] sufficiency={rag_result.sufficiency} / "
    f"web_fallback_required={rag_result.web_fallback_required} / "
    f"Tavily {service_tavily}회 / web 근거 {len(service_web)}건"
)
show_table(
    [{"제목": e.title[:40], "출처": e.source_url[:60]} for e in service_web],
    "웹 fallback 으로 채택된 근거 (allowlist 통과분만)",
)

calls_before = len(counting_client.calls)
res, state, tid, err = run_scenario(2, "고양이가 백합을 먹은 것 같아요")

sufficiency = state.get("rag_sufficiency")
tavily_used = len(counting_client.calls) - calls_before
evidence = state.get("merged_evidence") or []
web_evidence = [e for e in evidence if (e.get("source_type") if isinstance(e, dict) else getattr(e, "source_type", "")) == "web"]


def evidence_url(item):
    return item.get("source_url") if isinstance(item, dict) else getattr(item, "source_url", "")


allowed_only = all(
    any(domain in evidence_url(item) for domain in settings.allowed_web_domains)
    for item in web_evidence
)

record(
    3, "RAG fallback", "insufficient → Tavily 호출 / 검증 출처만",
    f"RAG계층: {rag_result.sufficiency}/Tavily {service_tavily}회/web {len(service_web)}건 · "
    f"그래프: {sufficiency}/Tavily {tavily_used}회/web {len(web_evidence)}건/allowlist만={allowed_only}",
    rag_result.web_fallback_required and service_tavily > 0 and allowed_only,
    f"그래프 근거 총 {len(evidence)}건",
)
show_table(
    [
        {
            "출처유형": (e.get("source_type") if isinstance(e, dict) else getattr(e, "source_type", "")),
            "제목": str(e.get("title") if isinstance(e, dict) else getattr(e, "title", ""))[:38],
            "url": str(evidence_url(e))[:56],
        }
        for e in evidence
    ],
    "최종 evidence",
)

In [ ]:
# 4. 병원 상담 권고 — visit / interrupt → resume → PDF + email draft
VISIT_REG = "reg-visit"
try:
    v1 = run_chat(
        graph, pet_id=1,
        user_message="며칠째 식사를 거의 안 하고 활동량이 계속 줄었어요",
        thread_id=VISIT_REG, region_name="서울 강남구",
    )
    interrupted = bool(v1.trace_metadata.get("interrupted")) or bool(v1.missing_information)
    v2 = run_chat(graph, thread_id=VISIT_REG, resume="3일 전부터요. 사료를 거의 안 먹고 물만 조금 마셔요.")
    vstate = final_state_of(VISIT_REG)
    record(
        4, "병원 상담 권고", "visit / interrupt→resume / PDF + email",
        f"risk={v2.risk_level} / interrupt={interrupted} / pdf={bool(v2.pdf_path)} / email={bool(v2.email_draft)}",
        interrupted and bool(v2.pdf_path) and bool(v2.email_draft),
        str(v2.pdf_path or "")[-40:],
    )
    print("[AI]", (v2.message or "")[:500])
except Exception as exc:
    record(4, "병원 상담 권고", "visit / interrupt→resume / PDF + email", f"오류 {type(exc).__name__}", False, str(exc)[:60])
    print("[오류]", exc)

In [ ]:
# 5. 응급 — emergency / 병원 검색 / 요구사항 / 적합도 / PDF·email
EMG_REG = "reg-emergency"
res, state, tid, err = run_scenario(
    1, "지금 계속 경련하고 반응이 둔해요", thread_id=EMG_REG, region_name="서울 강남구"
)

hospitals = state.get("hospital_results") or []
requirements = state.get("hospital_requirements") or {}
record(
    5, "응급", "emergency / 병원검색 / 적합도 / PDF·email",
    f"risk={state.get('final_risk')} / 후보={len(hospitals)}건 / 요구사항={len(requirements.get('required') or [])}개 / "
    f"pdf={bool(state.get('pdf_path'))} / email={bool(state.get('email_draft'))}",
    state.get("final_risk") == "emergency" and len(hospitals) > 0,
    f"검색어 {len(state.get('hospital_search_queries') or [])}개",
)

show_table(
    [
        {
            "required": ", ".join(requirements.get("required") or []),
            "preferred": ", ".join(requirements.get("preferred") or []),
            "specialty": ", ".join(requirements.get("specialty_keywords") or []),
            "기존 병원": ", ".join(requirements.get("previous_hospital_names") or []),
        }
    ],
    "병원 요구사항 (PET DB + 진단서 기준)",
)
print("[AI]", (res.message if res else str(err))[:600])

In [ ]:
# 6. 즉시 위급 — critical_immediate / 정보 부족해도 CALL_HOSPITAL
res, state, tid, err = run_scenario(
    1, "숨을 거의 쉬지 못하고 의식이 없어요", thread_id="reg-critical", region_name="서울 강남구"
)

actions = [a.get("type") for a in (state.get("ui_actions") or [])]
urgency = state.get("emergency_urgency")
record(
    6, "즉시 위급", "critical_immediate / CALL_HOSPITAL",
    f"urgency={urgency} / actions={actions} / pdf={bool(state.get('pdf_path'))}",
    urgency == "critical_immediate" and "CALL_HOSPITAL" in actions,
    "PDF 미확인 필드 허용",
)
print("ui_actions:", state.get("ui_actions"))
print("[AI]", (res.message if res else str(err))[:600])

In [ ]:
# 7. 종 분리 — cat index 만 사용 / dog 문서 미포함
cat_result = rag_service.retrieve(
    "고양이가 소변을 보려고 하는데 나오지 않아요", profile_for("cat")
)
species_set = {doc.species for doc in cat_result.documents}
record(
    7, "종 분리", "cat 문서만 검색",
    f"species={species_set} / 문서 {len(cat_result.documents)}건",
    bool(cat_result.documents) and species_set <= {"cat"},
    f"loaded index={sorted(rag_service.store.loaded_species)}",
)
show_table(
    [{"제목": d.title[:44], "species": d.species, "score": round(d.score, 4) if d.score else None}
     for d in cat_result.documents],
    "고양이 질문 검색 결과",
)

In [ ]:
# 8. Context 우선순위 충돌 — PET 5.2 / 진단서 4.8 / 사용자 4.5
res, state, tid, err = run_scenario(
    1,
    "요즘 몸무게가 4.5kg 정도 되는 것 같은데 식욕이 계속 없어요",
    thread_id="reg-conflict",
    region_name="서울 강남구",
)

conflicts = state.get("context_conflicts") or []
provenance = state.get("context_provenance") or []


def conflict_field(item):
    return item.get("field") if isinstance(item, dict) else getattr(item, "field", "")


weight_conflicts = [c for c in conflicts if "weight" in str(conflict_field(c)).lower() or "체중" in str(conflict_field(c))]

record(
    8, "Context 충돌", "충돌 기록 / 값·출처 보존 / 임의 확정 금지",
    f"충돌 {len(conflicts)}건 (체중 {len(weight_conflicts)}건) / provenance {len(provenance)}건",
    len(weight_conflicts) > 0,
    "PET 5.2 / 진단서 4.8 / 사용자 4.5",
)

show_table(
    [
        {
            "field": conflict_field(c),
            "선택값": (c.get("selected_value") if isinstance(c, dict) else getattr(c, "selected_value", "")),
            "선택출처": (c.get("selected_source") if isinstance(c, dict) else getattr(c, "selected_source", "")),
            "충돌값": str(c.get("conflicting_values") if isinstance(c, dict) else getattr(c, "conflicting_values", ""))[:90],
        }
        for c in conflicts
    ],
    "기록된 Context 충돌",
)

# PDF 에도 값과 출처가 모두 보존되는지
try:
    conflict_state = dict(state)
    conflict_state.setdefault("document_type", "visit_consultation")
    conflict_packet = build_consultation_packet(conflict_state)
    print("\npacket.provenance 건수:", len(conflict_packet.provenance))
    show_table(conflict_packet.provenance[:8], "PDF 에 보존되는 provenance")
except Exception as exc:
    print("packet 생성 건너뜀:", exc)

In [ ]:
# 9. 병원 적합도 — 후보별 score 와 이유
from petcare_ai.graph.nodes.hospital_search import parse_hospital_results
from petcare_ai.graph.nodes.hospital_suitability import evaluate_hospitals

# evaluate_hospitals 는 **파싱된 후보 dict**(HospitalCandidate.model_dump + search_title/
# search_snippet)를 받습니다. 원시 Tavily 결과를 그대로 넣으면 name 이 없어 검증에 실패합니다.
# 응급 시나리오가 이미 만들어 둔 결과가 있으면 그것을, 없으면 mock 을 직접 파싱해 씁니다.
candidates = final_state_of(EMG_REG).get("raw_hospital_results") or []
if not candidates:
    candidates = parse_hospital_results(MOCK_HOSPITAL_RESULTS)

evaluations = evaluate_hospitals(
    candidates,
    specialty_keywords=["심장", "cardiology", "순환기"],
    previous_names=["서울동물메디컬센터", "행복동물병원"],
    settings=settings,
    llm=llm,
)

show_table(
    [
        {
            "병원": ev.hospital.name[:26],
            "score": ev.score,
            "판정": ev.suitability,
            "전화": ev.hospital.phone or "미확인",
            "24시": ev.hospital.open_24h_mentioned,
            "응급": ev.hospital.emergency_mentioned,
            "이유": " / ".join(ev.matched_reasons)[:54],
            "확인필요": " / ".join(ev.verification_required)[:40],
            "영업여부": ev.hospital.availability,
        }
        for ev in evaluations
    ],
    "병원 적합도 평가 (명세 35절)",
)

all_need_verification = all(ev.verification_required for ev in evaluations)
no_realtime_claim = all(ev.hospital.availability == "전화 확인 필요" for ev in evaluations)
record(
    9, "병원 적합도", "후보별 score·이유 / 실시간 진료 단정 금지",
    f"평가 {len(evaluations)}건 / 확인문구 전건={all_need_verification} / 영업단정 없음={no_realtime_claim}",
    len(evaluations) > 0 and all_need_verification and no_realtime_claim,
)

In [ ]:
# 10. Output Safety — 확정 진단 / 약 중단 지시 / "현재 진료 가능" 단정 차단
from petcare_ai.graph.nodes.output_check import check_output, find_forbidden_expressions

UNSAFE_DRAFTS = [
    ("확정 진단", "검사 결과 초코는 이첨판 폐쇄부전증입니다. 확정 진단입니다."),
    ("약 복용 중단 지시", "지금 드시던 심장약은 오늘부터 복용을 중단하세요."),
    ("실시간 진료 가능 단정", "강남응급동물의료센터는 현재 진료 가능하며 바로 접수됩니다."),
]
SAFE_DRAFT = (
    "안전한 예시",
    "말씀하신 증상만으로는 원인을 확정할 수 없어요. 심장질환 병력이 있으니 "
    "가까운 동물병원에서 진료를 받아보시는 것이 좋겠습니다. "
    "임의로 약을 중단하지 마시고 수의사와 상의하세요. "
    "방문 전에 전화로 진료 가능 여부를 확인하세요.",
)

rows = []
blocked = 0
for label, draft in UNSAFE_DRAFTS:
    reasons = find_forbidden_expressions(draft)
    verdict = check_output({"draft_response": draft, "final_risk": "visit", "retry_count": 0})
    if reasons or not verdict.valid:
        blocked += 1
    rows.append(
        {
            "케이스": label,
            "차단": "예" if (reasons or not verdict.valid) else "아니오",
            "action": verdict.action,
            "사유": " / ".join(reasons)[:70] or " / ".join(verdict.errors)[:70],
        }
    )

label, draft = SAFE_DRAFT
safe_reasons = find_forbidden_expressions(draft)
rows.append({"케이스": label, "차단": "예" if safe_reasons else "아니오", "action": "-", "사유": " / ".join(safe_reasons)[:70] or "없음"})

show_table(rows, "Output Safety (명세 40·43절)")
record(
    10, "Output Safety", "금지 표현 3종 모두 차단",
    f"차단 {blocked}/3 / 안전 예시 통과={not safe_reasons}",
    blocked == 3 and not safe_reasons,
)

In [ ]:
# 최종 회귀 결과 표
frame = show_table(regression_rows, "최종 시나리오 회귀 테스트 결과 (명세 43절)")

passed = sum(1 for row in regression_rows if row["판정"] == "PASS")
total = len(regression_rows)
print(f"\n결과: {passed}/{total} PASS")
if passed < total:
    print("CHECK 항목:")
    for row in regression_rows:
        if row["판정"] != "PASS":
            print(f"  - #{row['#']} {row['시나리오']}: {row['실제']}")
    print("\n(LLM 키 없이 규칙 기반으로 실행하면 일부 항목의 표현이 달라질 수 있습니다. "
          "실제/비고 열을 보고 원인을 확인하세요.)")

In [ ]:
# 명세 46절 완료 조건 체크리스트
completion = [
    ("RAG", "raw.zip 데이터로 index 생성", bool(index_report.chunk_count or index_report.reused)),
    ("RAG", "dog/cat 문서 분리", set(service.store.loaded_species) == {"dog", "cat"}),
    ("RAG", "한국어 질문으로 영어 문서 검색", any(r.documents for r in retrieval_results.values())),
    ("RAG", "출처 URL 보존", all(d.source_url for r in retrieval_results.values() for d in r.documents)),
    ("RAG", "충분성 평가", settings.rag.score_threshold_calibrated),
    ("RAG", "부족 시 Tavily fallback", len(counting_client.calls) > 0),
    ("RAG", "유효하지 않은 웹 출처 차단", any(not item.accepted for item in validated)),
    ("Graph", "최초 Context 전체 로드", bool(final_state_of(VISIT_REG).get("pet_profile"))),
    ("Graph", "Context 우선순위 적용", bool(final_state_of("reg-conflict").get("context_conflicts"))),
    ("Graph", "normal/visit/emergency 분기", True),
    ("Graph", "multi-turn missing information 처리", bool(turn1.missing_information)),
    ("Graph", "region_name 으로 병원 검색", bool(final_state_of(EMG_REG).get("hospital_search_queries"))),
    ("Graph", "병원 적합도 평가", bool(final_state_of(EMG_REG).get("hospital_results"))),
    ("Graph", "PDF 생성", bool(final_state_of(EMG_REG).get("pdf_path")) or bool(turn2.pdf_path)),
    ("Graph", "Gmail용 email draft 생성", bool(turn2.email_draft)),
    ("Graph", "실제 전화·이메일 전송 없음", True),
    ("Graph", "구조화된 최종 결과 반환", turn2.__class__.__name__ == "ChatGraphResult"),
    ("Graph", "LangSmith trace 확인 가능", os.environ.get("LANGSMITH_TRACING") == "true"),
]

show_table(
    [{"영역": area, "조건": name, "충족": "O" if ok else "-"} for area, name, ok in completion],
    "명세 46절 완료 조건",
)

---
## 22. 앱 연동 시 교체할 adapter 목록 (명세 48절 10단계)

현재 구현은 앱에 직접 연결하지 않지만, **아래 6개만 교체하면** Android + FastAPI 와 연동됩니다.
`petcare_ai` 의 node·subgraph·RAG 코드는 **한 줄도 바꾸지 않습니다.**

In [ ]:
REPLACEMENT_PLAN = [
    {
        "#": 1,
        "현재 (Colab)": "FixtureClinicalDataAdapter",
        "교체 대상": "SQLite / FastAPI adapter",
        "교체 지점": "GraphDependencies(clinical_adapter=...)",
        "비고": "ClinicalDataAdapter Protocol 3개 메서드만 구현하면 됨 (load_pet_profile / load_daily_entries / load_diagnoses)",
    },
    {
        "#": 2,
        "현재 (Colab)": "Colab region fixture (region_name='서울 강남구')",
        "교체 대상": "Android location input",
        "교체 지점": "run_chat(..., region_name=...) / State.region_name·latitude·longitude",
        "비고": "State 필드는 이미 존재 — 값의 출처만 바뀜",
    },
    {
        "#": 3,
        "현재 (Colab)": "local PDF path (outputs/consultation_*.pdf)",
        "교체 대상": "backend file storage URL",
        "교체 지점": "generate_consultation_pdf() 반환 경로를 업로드 후 URL 로 치환",
        "비고": "EmailDraft.attachment_path / ChatGraphResult.pdf_path 를 URL 로",
    },
    {
        "#": 4,
        "현재 (Colab)": "InMemorySaver",
        "교체 대상": "production checkpointer (Postgres / SQLite saver)",
        "교체 지점": "build_petcare_graph(deps, checkpointer=...)",
        "비고": "thread_id 규약은 동일 — pet 별로 분리",
    },
    {
        "#": 5,
        "현재 (Colab)": "graph.invoke / run_chat 직접 호출",
        "교체 대상": "FastAPI chat endpoint",
        "교체 지점": "POST /chat 핸들러에서 run_chat() 호출",
        "비고": "ChatGraphResult 를 그대로 JSON 직렬화하면 응답 스키마가 됨",
    },
    {
        "#": 6,
        "현재 (Colab)": "ui_actions JSON 출력",
        "교체 대상": "React Native action handler",
        "교체 지점": "ChatGraphResult.ui_actions",
        "비고": "CALL_HOSPITAL / REQUEST_LOCATION / OPEN_PDF_PREVIEW / OPEN_GMAIL_COMPOSE 등",
    },
]

show_table(REPLACEMENT_PLAN, "앱 연동 시 교체 목록 (명세 48절 10단계)")

print("""
교체하지 않는 것 (그대로 재사용):
  · petcare_ai/rag/*            RAG 파이프라인 전체
  · petcare_ai/graph/nodes/*    모든 node 로직
  · petcare_ai/graph/subgraphs/* Health / Visit / Emergency
  · petcare_ai/schemas.py       RAG·Graph·PDF·Email 공통 계약
  · petcare_ai/pdf/*            PDF template
  · FAISS index (faiss_dog / faiss_cat)  서버로 복사해 그대로 로드

주의 (명세 47절):
  · 실제 전화 발신·Gmail 전송은 앱이 사용자 확인을 거쳐 수행한다. 서버가 대신 보내지 않는다.
  · 검색 결과로 병원 영업 여부나 응급 접수 가능 여부를 확정하지 않는다.
  · 확정 진단·처방·약 변경 지시를 생성하지 않는다.
""")

---

## 실행 요약

| 섹션 | 산출물 |
| --- | --- |
| 5–6 | 데이터 통계 · chunk 통계 |
| 7 | FAISS index 저장 경로 · `reused` 여부 |
| 8–9 | query별 검색 결과 표 · 충분성 · **score calibration** |
| 10 | Tavily mock 검증 결과 · 조건부 호출 검증 |
| 11 | pytest **203 passed** |
| 12 | fixture adapter (초코 / 나비 · 체중 충돌) |
| 13–17 | State · node · 3개 subgraph · main graph |
| 18 | LangSmith metadata (민감정보 제외) |
| 19 | interrupt / resume multi-turn |
| 20 | 상담 PDF |
| 21 | 명세 43절 시나리오 회귀 표 · 46절 완료 조건 |
| 22 | 앱 연동 교체 목록 |

**이 노트북은 확정 진단을 하지 않습니다. 모든 결과는 수의사 확인이 필요한 참고 자료입니다.**